# 01 ? Document Ingestion

**Objective:** Test the project's real PDF and DOCX extraction pipeline.

This notebook uses `app.ingestion.loader.load_document`. It reports extraction statistics rather than displaying course-document content.

**Supported formats:** PDF, DOCX, PPTX, TXT and Markdown.

In [2]:
from pathlib import Path
import sys
import tempfile

# Automatically locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "app").exists():
    raise FileNotFoundError("Project root containing app/ was not found.")

sys.path.insert(0, str(PROJECT_ROOT))

from docx import Document
from app.ingestion.loader import load_document

DOCS_DIR = PROJECT_ROOT / "docs"

print("Project root:", PROJECT_ROOT)
print("Docs directory exists:", DOCS_DIR.exists())
print("App import successful!")

Project root: C:\portfolio 1\IEC-Production-RAG--Services
Docs directory exists: True
App import successful!


## 1. Extract a PDF

The notebook selects `main_notes.pdf` when available. Only page counts and text lengths are displayed.

In [3]:
pdf_path = DOCS_DIR / "main_notes.pdf"

if not pdf_path.exists():
    pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError("No PDF file found in docs/.")
    pdf_path = pdf_files[0]

pdf_pages = load_document(str(pdf_path))

print("PDF filename:", pdf_path.name)
print("Extracted non-empty pages:", len(pdf_pages))
print("Total extracted characters:", sum(len(p["text"]) for p in pdf_pages))
print("First page number:", pdf_pages[0]["page"])
print("Extraction successful:", bool(pdf_pages))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FRGUZY+Helvetica', '/FontDescriptor': IndirectObject(697, 0, 2213593763792), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 117, '/Widths': [278, 0, 0, 0, 556, 0, 0, 0, 333, 333, 0, 0, 0, 0, 0, 0, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 500, 0, 556, 278, 556, 556, 222, 0, 0, 0, 0, 556, 556, 556, 556, 333, 500, 278, 556], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KZCTHM+Times-Roman', '/FontDescriptor': IndirectObject(707, 0, 2213593763792), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 121, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 333, 333, 0, 0,

PDF filename: main_notes.pdf
Extracted non-empty pages: 278
Total extracted characters: 469207
First page number: 1
Extraction successful: True


## 2. Test DOCX extraction with a synthetic document

A temporary DOCX is generated for this test. It contains no course or third-party material.

In [4]:
with tempfile.TemporaryDirectory() as temp_dir:
    synthetic_docx = Path(temp_dir) / "synthetic_test.docx"

    document = Document()
    document.add_heading("Synthetic RAG Test Document", level=1)
    document.add_paragraph(
        "Retrieval augmented generation retrieves relevant document "
        "chunks before producing a grounded response."
    )
    document.save(synthetic_docx)

    docx_pages = load_document(str(synthetic_docx))

    print("DOCX pages/sections extracted:", len(docx_pages))
    print("Extracted characters:", sum(len(p["text"]) for p in docx_pages))
    print("DOCX extraction successful:", bool(docx_pages))

DOCX pages/sections extracted: 1
Extracted characters: 131
DOCX extraction successful: True


## 3. Ingestion observations

- PDF extraction preserves page numbers.
- DOCX paragraphs are combined into one logical page by the current loader.
- Scanned/image-only PDFs may require OCR; OCR is not implemented in the current loader.
- Keep source documents out of public GitHub unless redistribution is permitted.